# Step 2. Data Understanding

Step 1 framed the problem. Step 2 checks that the data can carry that problem. It also builds the reference sheet that every later step reads from.

Three questions guide the work.

1. Where does the data come from? Can we trust it?
2. What is in the table? Is it clean?
3. What kind of thing is each column? Which columns matter for fairness?

This work feeds two later steps. Step 3 cleans and prepares the data. Step 5 checks it for bias.

In [1]:
import sys
from pathlib import Path

# I add the repo root to the path so src/ is importable from notebooks/.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from src.data import (
    load_raw, load_binary,
    CONTINUOUS, BINARY_FLAGS, NOMINAL_CODED, COUNT_ORDINAL, LEAKAGE,
    SENSITIVE, TARGET,
)

df = load_raw()      # all 37 columns, names cleaned
bdf = load_binary()  # Dropout vs Graduate, with a 0/1 dropout column
print("raw", df.shape, "| binary", bdf.shape)

raw (4424, 37) | binary (3630, 38)


## 1. Where the Data Comes From

The data is a public dataset called Predict Students' Dropout and Academic Success. It comes from UCI. Realinho and colleagues at the Polytechnic Institute of Portalegre built it. It carries a CC BY 4.0 license. That means anyone can reuse it for free as long as they give credit. Its DOI is 10.24432/C5MC89. The full citation, funding, and license all sit in data/README.md.

The file pulls several separate school databases into one table. Each row is one student. The people who made the dataset say they cleaned it before release. They removed odd values, extreme values, and missing entries. That fact matters for the next section. The data is clean because they cleaned it. Raw school records are not clean on their own.

## 2. Dataset Overview

This is a first look at the whole table. We check its size. We check the type of each column. We look for missing values and repeated rows. We see how the outcome splits. We also check the range of the real number columns.

In [2]:
print("shape", df.shape)
print()
print("dtypes")
print(df.dtypes.value_counts())
print()
print("missing values total", int(df.isna().sum().sum()))
print("duplicate rows", int(df.duplicated().sum()))
print()
print("target split")
print(df[TARGET].value_counts())

shape (4424, 37)

dtypes
int64      29
float64     7
str         1
Name: count, dtype: int64

missing values total 0
duplicate rows 0

target split
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [3]:
# I check the ranges and spread of the six true numbers. Order and distance are real here.
df[CONTINUOUS].describe().round(1)

,Previous qualification (grade),Admission grade,Age at enrollment,Unemployment rate,Inflation rate,GDP
count,4424.0,4424.0,4424.0,4424.0,4424.0,4424.0
mean,132.6,127.0,23.3,11.6,1.2,0.0
std,13.2,14.5,7.6,2.7,1.4,2.3
min,95.0,95.0,17.0,7.6,-0.8,-4.1
25%,125.0,117.9,19.0,9.4,0.3,-1.7
50%,133.1,126.1,20.0,11.1,1.4,0.3
75%,140.0,134.8,25.0,13.9,2.6,1.8
max,190.0,190.0,70.0,16.2,3.7,3.5


There are no missing cells and no repeated rows. This is not luck. It comes from the cleaning the providers did, which we noted above. So Step 3 does not clean the data again. Instead it checks the ranges and keeps the real extreme values, like the ages of older students.

The outcome splits three ways. Graduate has 2209 students. Dropout has 1421. Enrolled has 794. The binary frame is the table cut down to just two outcomes, Dropout and Graduate. It drops the Enrolled group and works with 3630 students.

The ranges look sound. Grades fall between 0 and 200. The one long tail is age, which runs from 17 up to 70 as mature students enrol, against a median of 20. Those extreme ages are the only outliers here, and they are real students rather than data errors, so no rows are cut. Cutting that tail would delete exactly the group that drops out most, and Step 3 engineers `mature entry` to make use of it instead. The full distributions, one per number, are examined in Step 3, where the shapes turn out to be skewed rather than bell-shaped.

## 3. Feature Types

The 37 columns are not all the same kind of thing. They fall into a few types. Each type needs different handling later. Naming the types now helps us avoid a common mistake in Step 3.

Four types make up the inputs the model learns from.

True numbers. Six columns hold a real quantity. Order and distance both mean something here. Admission grade and age are two examples.

Yes or no flags. Eight columns store a simple 1 or 0. Gender, scholarship holder, and debtor are examples.

Coded categories. Nine columns use a whole number to stand for a category. Course and mother's qualification are examples. The number is just a label. It is not an amount. A course coded 9500 is not bigger than one coded 33. It is simply a different course. So it would be wrong to treat these as real numbers. We should not scale them or measure plain correlation on them. Step 3 handles them as categories instead.

Ordered count. This is one column, application order. It ranks a student's choices from 0 for first choice to 9 for last choice.

Twelve more columns record how students did in their first and second semesters. Step 1 holds these out as leakage. Leakage means a clue that would not be known yet at the moment we make the prediction. The Target column holds the outcome we want to predict.

In [4]:
# I confirm the grouping covers all 37 columns.
groups = {
    "true number": CONTINUOUS,
    "yes or no flag": BINARY_FLAGS,
    "coded category": NOMINAL_CODED,
    "ordered count": COUNT_ORDINAL,
    "curricular (leakage)": LEAKAGE,
    "target": [TARGET],
}
for name, cols in groups.items():
    print(f"{name:22} {len(cols)}")
print("total", sum(len(c) for c in groups.values()))

true number            6
yes or no flag         8
coded category         9
ordered count          1
curricular (leakage)   12
target                 1
total 37


## 4. The Data Dictionary

The dictionary lists every column. For each one it shows the type, the range or the set of values, the number of unique values, and whether it is a sensitive attribute. A sensitive attribute is a personal trait we must watch for fairness. The next cell builds the dictionary straight from the data. So it always matches the data and cannot fall out of step with it. A grouped, plain language version comes after it for easy reading.

In [5]:
# I build the dictionary from the data, so it matches the file exactly.
group_lookup = {c: name for name, cols in groups.items() for c in cols}

def describe_range(col):
    s = df[col]
    if col in CONTINUOUS or col in LEAKAGE:
        return f"{s.min():g} to {s.max():g}"
    if col in BINARY_FLAGS:
        return "0 or 1"
    if col in NOMINAL_CODED:
        return f"{s.nunique()} codes"
    if col in COUNT_ORDINAL:
        return f"{int(s.min())} to {int(s.max())}"
    if col == TARGET:
        return ", ".join(map(str, s.unique()))
    return ""

data_dict = pd.DataFrame([
    {
        "column": c,
        "group": group_lookup.get(c, "other"),
        "dtype": str(df[c].dtype),
        "unique": int(df[c].nunique()),
        "range or values": describe_range(c),
        "missing": int(df[c].isna().sum()),
        "sensitive": "yes" if c in SENSITIVE else "",
    }
    for c in df.columns
])
data_dict

,column,group,dtype,unique,range or values,missing,sensitive
0,Marital status,coded category,int64,6,6 codes,0,
1,Application mode,coded category,int64,18,18 codes,0,
2,Application order,ordered count,int64,8,0 to 9,0,
3,Course,coded category,int64,17,17 codes,0,
4,Daytime/evening attendance,yes or no flag,int64,2,0 or 1,0,
5,Previous qualification,coded category,int64,17,17 codes,0,
6,Previous qualification (grade),true number,float64,101,95 to 190,0,
7,Nacionality,coded category,int64,21,21 codes,0,
8,Mother's qualification,coded category,int64,29,29 codes,0,
9,Father's qualification,coded category,int64,34,34 codes,0,


Here are the same columns grouped, with their plain meaning. The table above holds the exact types and ranges.

| group | columns | meaning |
|---|---|---|
| true number | Admission grade, Previous qualification (grade), Age at enrollment, Unemployment rate, Inflation rate, GDP | real quantities where order and distance matter |
| yes or no flag | Gender, Scholarship holder, Debtor, Tuition fees up to date, Displaced, Educational special needs, International, Daytime/evening attendance | stored as 1 or 0 |
| coded category | Course, Marital status, Application mode, Previous qualification, Nacionality, Mother's and Father's qualification, Mother's and Father's occupation | integer codes standing for categories, labels not amounts |
| ordered count | Application order | rank from 0 first choice to 9 last choice |
| curricular, leakage | twelve first and second semester records | performance after enrollment, held out of the model |
| target | Target | Dropout, Graduate, Enrolled, with Enrolled dropped for the binary frame |

The sensitive attributes for the Step 5 audit are Gender, Age at enrollment, Scholarship holder, and Debtor. Tuition fees up to date is watched too. The next section shows why.

## 5. Sensitive Attributes, and a First Look at Disparity

Four traits drive the fairness work in Step 5. They are gender, age at enrollment, scholarship holder, and debtor. The next cell shows the dropout rate inside each group. It compares each group to the overall rate, which is near 39 percent. This is the starting point that the Step 5 audit builds on.

In [6]:
bdf["age band"] = pd.cut(
    bdf["Age at enrollment"], bins=[16, 20, 23, 30, 100],
    labels=["17 to 20", "21 to 23", "24 to 30", "31 plus"],
)

overall = bdf["dropout"].mean()
print("overall dropout rate", round(overall * 100, 1), "percent")
print()

def show_rate(col, labels=None):
    g = bdf.groupby(col, observed=True)["dropout"].agg(["mean", "size"])
    print(col)
    for idx, row in g.iterrows():
        name = labels.get(idx, idx) if labels else idx
        print(f"  {str(name):16} {row['mean'] * 100:5.1f} percent   n={int(row['size'])}")
    print()

show_rate("Gender", {1: "male", 0: "female"})
show_rate("Scholarship holder", {1: "scholarship", 0: "no scholarship"})
show_rate("Debtor", {1: "debtor", 0: "not debtor"})
show_rate("Tuition fees up to date", {1: "up to date", 0: "not up to date"})
show_rate("age band")

overall dropout rate 39.1 percent

Gender
  female            30.2 percent   n=2381
  male              56.1 percent   n=1249

Scholarship holder
  no scholarship    48.4 percent   n=2661
  scholarship       13.8 percent   n=969

Debtor
  not debtor        34.5 percent   n=3217
  debtor            75.5 percent   n=413

Tuition fees up to date
  not up to date    94.0 percent   n=486
  up to date        30.7 percent   n=3144

age band
  17 to 20          26.1 percent   n=2080
  21 to 23          40.6 percent   n=473
  24 to 30          66.5 percent   n=499
  31 plus           61.4 percent   n=578



### Reading the Gaps

The gaps are real and large. Men drop out far more than women, 56 percent against 30 percent. Students without a scholarship drop out far more than students who hold one, 48 percent against 14 percent. Students with debt drop out far more than students without debt, 76 percent against 35 percent. Older students drop out more than the youngest ones. The rate is near 66 percent in the 24 to 30 age band. It is 26 percent in the 17 to 20 band.

A model trained on this data can carry these gaps into the students it flags. So Step 5 measures the gaps and works to shrink them.

One column stands apart. It is tuition fees up to date. Students who are not up to date drop out almost every time, 94 percent. This is not a normal background fact. A student who has stopped paying is often a student who is already leaving. So this flag sits very close to the outcome. It acts like the semester records that Step 1 held out. We keep it for now, but we watch it. Step 3 tests the model both with it and without it. That way its strong effect is measured out in the open, not hidden.

## What Step 2 Settles

Five things carry into Step 3.

1. The dataset is cited and licensed. The full record sits in data/README.md.
2. There are no missing values and no repeated rows. The providers cleaned it to this state.
3. The 37 columns fall into four input types. Twelve more curricular columns are held out as leakage.
4. The coded categories are labels, not amounts. They need category handling, not scaling.
5. All four sensitive attributes show real gaps. Tuition status acts like a near-outcome signal that we must watch.

Step 3 builds on this base. It cleans the data, explores it, and makes new features. It sends each feature type down the right path.